# 문화콘텐츠 스토리 데이터 전처리

**목적**: 메타데이터 RAG 및 구조 설계(통계 가이드) 두 방식 모두에 사용할 전처리 파일 생성  
**참고**: `문화콘텐츠.ipynb`, `메타데이터RAG_vs_구조설계_비교계획.ipynb`, `데이터셋활용계획_1.ipynb`

## 데이터 구조 요약

```
01-1.정식개방데이터/
├── Training/02.라벨링데이터/  TL_01.영화(1,327) TL_02.시리즈(1,297) TL_03.소설(168) TL_04.만화(368)
└── Validation/02.라벨링데이터/ VL_01.영화(104)  VL_02.시리즈(205)  VL_03.소설(35)  VL_04.만화(57)
```

- 작품 하나 = JSON 파일 1개 (Training **또는** Validation 중 한 곳에만 존재)
- **전체 작품을 얻으려면 Training + Validation을 반드시 합산해야 함**
- 총 작품 수: 영화 1,431 / 시리즈 1,502 / 소설 203 / 만화 425 = **3,561개**

## 스토리 프레임워크

| 프레임워크 | stage 레이블 | 작품 수 | 주요 장르 |
|-----------|-------------|--------|---------|
| 스토리헬퍼 15단계 | 영문 (Opening Salvo 등) | ~3,512개 | 전 장르 |
| 영웅의 12단계 | 한국어 (평범한 세계 등) | ~49개 | 판타지·액션 집중 |

- 두 프레임워크는 stage 레이블 체계가 달라 **분리 처리** (stage 통계만 분리, unit_motif·theme은 통합)
- 영웅의 12단계 작품도 **살려서 활용** — 판타지 장르 캐릭터 채팅에 유용

## 전처리 출력 파일

### 문화콘텐츠 스토리 데이터 (AI Hub 145)

| 파일 | 용도 |
|------|------|
| `output/scene_metadata_index.json` | 메타데이터 RAG용 — 씬(unit) 단위 플랫 인덱스 (`framework` 필드 포함) |
| `output/work_metadata_index.json` | 작품 단위 메타데이터 인덱스 (`framework` 필드 포함) |
| `output/genre_stats.json` | 구조 설계용 — 장르별 통계 (stage는 `top_stages_storyhelper` / `top_stages_hero_journey`로 분리) |

### 동아시아 고전 데이터 (AI Hub 70)

> 유저가 "조선", "중국 무협", "일본 설화" 등 **고전 장르를 명시적으로 선택했을 때만** 보조 참조

| 파일 | 용도 |
|------|------|
| `output/classic_paragraph_index.json` | 고전 RAG용 — 단락 단위 플랫 인덱스 (국가/장르/모티프/공간 등 메타데이터) |
| `output/classic_country_stats.json` | 국가별 장르·모티프·공간 통계 |

## 0. 환경 설정

In [11]:
import json
import os
from pathlib import Path
from collections import Counter, defaultdict

# ── 문화콘텐츠 경로 설정 ─────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).parent / "문화콘텐츠 스토리 데이터/145.다양한_문화콘텐츠_스토리_데이터/01-1.정식개방데이터" \
             if "__file__" in dir() else \
             Path("C:/Users/User/Desktop/Github/Dive.ai/문화콘텐츠 스토리 데이터/145.다양한_문화콘텐츠_스토리_데이터/01-1.정식개방데이터")

# ── 동아시아 고전 경로 설정 ───────────────────────────────────────────────────
CLASSIC_BASE_DIR = Path("C:/Users/User/Desktop/Github/Dive.ai/동아시아 고전 데이터/70.동아시아_고전_스토리_데이터/3.개방데이터/1.데이터")

OUTPUT_DIR = Path("C:/Users/User/Desktop/Github/Dive.ai/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 문화콘텐츠 카테고리 매핑 ─────────────────────────────────────────────────
CATEGORIES = {
    "01": "영화",
    "02": "시리즈",
    "03": "소설",
    "04": "만화",
}

# ── 동아시아 고전 국가 매핑 ───────────────────────────────────────────────────
COUNTRY_MAP    = {"01": "한국", "02": "중국", "03": "일본"}
COUNTRY_GROUPS = {"한국": 7, "중국": 8, "일본": 8}  # 국가별 그룹(폴더) 수

# ── 고전 장르 판별 (전 파이프라인 공통 사용) ─────────────────────────────────
CLASSIC_GENRES = {"조선", "중국무협", "일본설화", "한국설화", "동아시아고전"}

def is_classic_genre(background: str) -> bool:
    """유저가 선택한 배경이 고전 장르인지 판별."""
    return any(tag in background for tag in CLASSIC_GENRES)

def get_framework(structure: str) -> str:
    """작품의 structure 필드로 스토리 프레임워크를 판별.

    Returns
    -------
    'hero_journey'  : 영웅의 12단계 (stage 레이블이 한국어)
    'storyhelper'   : 스토리헬퍼 15단계 (stage 레이블이 영문, 기본값)
    """
    if structure and "영웅" in structure:
        return "hero_journey"
    return "storyhelper"

print("BASE_DIR 존재 여부:", BASE_DIR.exists())
print("CLASSIC_BASE_DIR 존재 여부:", CLASSIC_BASE_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR 존재 여부: True
CLASSIC_BASE_DIR 존재 여부: True
OUTPUT_DIR: C:\Users\User\Desktop\Github\Dive.ai\output


## 1. 전체 라벨링 데이터 로드

Training + Validation 두 폴더를 모두 순회하여 전체 작품의 라벨링 JSON을 합산합니다.

In [12]:
def iter_all_labels():
    """Training + Validation 전체 라벨링 데이터를 순회하는 제너레이터.
    
    Yields
    ------
    dict : 라벨링 JSON 객체 (category_code 필드 추가)
    """
    for split in ["Training", "Validation"]:
        prefix = "TL" if split == "Training" else "VL"
        for cat_code, cat_name in CATEGORIES.items():
            lbl_dir = BASE_DIR / split / "02.라벨링데이터" / f"{prefix}_{cat_code}.{cat_name}"
            if not lbl_dir.exists():
                print(f"[경고] 폴더 없음: {lbl_dir}")
                continue
            for json_file in sorted(lbl_dir.glob("*.json")):
                with open(json_file, encoding="utf-8") as f:
                    data = json.load(f)
                data["_category_code"] = cat_code   # 편의상 카테고리 코드 추가
                data["_split"] = split               # Training / Validation 출처 추가
                yield data


# 전체 로드 및 카운트 확인
all_works = list(iter_all_labels())

count_by_cat = Counter(w["_category_code"] for w in all_works)
count_by_split = Counter(w["_split"] for w in all_works)

print(f"총 작품 수: {len(all_works):,}개")
print("\n카테고리별:")
for code, name in CATEGORIES.items():
    print(f"  {name}: {count_by_cat[code]:,}개")
print("\nsplit별:")
for split, cnt in count_by_split.items():
    print(f"  {split}: {cnt:,}개")

총 작품 수: 3,561개

카테고리별:
  영화: 1,431개
  시리즈: 1,502개
  소설: 203개
  만화: 425개

split별:
  Training: 3,160개
  Validation: 401개


## 2. 작품 단위 메타데이터 인덱스 생성

각 작품의 작품 레벨 정보를 정리합니다.

In [13]:
work_index = []

for work in all_works:
    framework = get_framework(work.get("structure", ""))
    work_index.append({
        "id":             work.get("id"),
        "type":           work.get("type"),
        "category_code":  work["_category_code"],
        "category_name":  CATEGORIES[work["_category_code"]],
        "split":          work["_split"],
        "title":          work.get("title"),
        "year":           work.get("year"),
        "genre":          work.get("genre", []),
        "theme":          work.get("theme"),
        "concept":        work.get("concept"),
        "structure":      work.get("structure"),
        "framework":      framework,           # 'storyhelper' | 'hero_journey'
        "motif":          work.get("motif"),
        "main_character": work.get("main_character"),
        "conflict":       work.get("conflict"),
        "characters":     work.get("characters", []),
        "unit_count":     len(work.get("units", [])),
    })

# 저장
out_path = OUTPUT_DIR / "work_metadata_index.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(work_index, f, ensure_ascii=False, indent=2)

# 프레임워크별 작품 수 확인
fw_counter = Counter(w["framework"] for w in work_index)
print(f"작품 인덱스 저장 완료: {out_path}")
print(f"총 {len(work_index):,}개 작품")
print(f"\n프레임워크별:")
print(f"  storyhelper  (스토리헬퍼 15단계): {fw_counter['storyhelper']:,}개")
print(f"  hero_journey (영웅의 12단계):     {fw_counter['hero_journey']:,}개")
print("\n샘플:")
print(json.dumps(work_index[0], ensure_ascii=False, indent=2))

작품 인덱스 저장 완료: C:\Users\User\Desktop\Github\Dive.ai\output\work_metadata_index.json
총 3,561개 작품

프레임워크별:
  storyhelper  (스토리헬퍼 15단계): 3,512개
  hero_journey (영웅의 12단계):     49개

샘플:
{
  "id": "01_0017",
  "type": "movie",
  "category_code": "01",
  "category_name": "영화",
  "split": "Training",
  "title": "아이들이 보인다",
  "year": 2003,
  "genre": [
    "스릴러",
    "공포(호러)",
    "드라마"
  ],
  "theme": "발견",
  "concept": "C001은 퇴근길에 이상한 두 자매를 발견한 이후, 집 부엌 테이블에서 자꾸 귀신을 본다.",
  "structure": "스토리헬퍼 15단계",
  "framework": "storyhelper",
  "motif": "갑작스런 사고",
  "main_character": "방어적임",
  "conflict": "C001은 집에 보이는 귀신이 자기 눈에만 보이다가 C002의 눈에도 보인다는 것을 알고 혼란스러워한다.",
  "characters": [
    "C001",
    "C002",
    "C003",
    "C004",
    "C005",
    "C006",
    "C007",
    "C008",
    "C009",
    "C010"
  ],
  "unit_count": 40
}


## 3. 씬(unit) 단위 메타데이터 인덱스 생성 — 메타데이터 RAG용

각 씬의 구조적 정보를 플랫하게 펼쳐서 인덱싱합니다.  
벡터 DB 구축 전 단계에서는 이 파일을 장르/모티프로 직접 필터링하여 시뮬레이션합니다.

In [14]:
scene_index = []

for work in all_works:
    work_id       = work.get("id")
    cat_code      = work["_category_code"]
    genre         = work.get("genre", [])
    theme         = work.get("theme")
    work_motif    = work.get("motif")
    main_char     = work.get("main_character")
    framework     = get_framework(work.get("structure", ""))  # 작품 레벨에서 상속

    for unit in work.get("units", []):
        scene_index.append({
            # ── 작품 레벨 컨텍스트 ──────────────────────────────────────────
            "work_id":        work_id,
            "category_code":  cat_code,
            "category_name":  CATEGORIES[cat_code],
            "genre":          genre,           # 장르 리스트 (필터링 키)
            "theme":          theme,           # 테마 (필터링 키)
            "work_motif":     work_motif,      # 작품 전체 모티프
            "main_character": main_char,       # 주인공 성격 유형
            "framework":      framework,       # 'storyhelper' | 'hero_journey'
            # ── 씬(unit) 레벨 정보 ──────────────────────────────────────────
            "scene_id":       unit.get("id"),
            "stage":          unit.get("stage"),       # 스토리 단계 (RAG 핵심 키)
            "unit_motif":     unit.get("unit_motif"),  # 씬 모티프 (RAG 핵심 키)
            "storyline":      unit.get("storyline"),   # 씬 요약 (LLM 주입 텍스트)
            "causality":      unit.get("causality"),   # 씬 간 인과관계 (프롬프트 조건부 사용)
            "characters":     unit.get("characters", []),
            "script_count":   len(unit.get("story_scripts", [])),
            # ── 감정 집계 ────────────────────────────────────────────────────
            "dominant_emotions": [
                e for e, _ in Counter(
                    s["emotion"] for s in unit.get("story_scripts", []) if s.get("emotion")
                ).most_common(2)
            ],
        })

# 저장
out_path = OUTPUT_DIR / "scene_metadata_index.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(scene_index, f, ensure_ascii=False, indent=2)

# 프레임워크별 씬 수 확인
fw_scene_counter = Counter(s["framework"] for s in scene_index)
print(f"씬 인덱스 저장 완료: {out_path}")
print(f"총 {len(scene_index):,}개 씬")
print(f"\n프레임워크별:")
print(f"  storyhelper  (스토리헬퍼 15단계): {fw_scene_counter['storyhelper']:,}개")
print(f"  hero_journey (영웅의 12단계):     {fw_scene_counter['hero_journey']:,}개")
print("\n샘플:")
print(json.dumps(scene_index[0], ensure_ascii=False, indent=2))

씬 인덱스 저장 완료: C:\Users\User\Desktop\Github\Dive.ai\output\scene_metadata_index.json
총 89,851개 씬

프레임워크별:
  storyhelper  (스토리헬퍼 15단계): 88,424개
  hero_journey (영웅의 12단계):     1,427개

샘플:
{
  "work_id": "01_0017",
  "category_code": "01",
  "category_name": "영화",
  "genre": [
    "스릴러",
    "공포(호러)",
    "드라마"
  ],
  "theme": "발견",
  "work_motif": "갑작스런 사고",
  "main_character": "방어적임",
  "framework": "storyhelper",
  "scene_id": "01_0017_01",
  "stage": "Opening Salvo",
  "unit_motif": "긴장감",
  "storyline": "C001은 지하철 안에서 졸다가 깨어 내리는데 두 여자아이를 발견하고 섬찟한다.",
  "causality": null,
  "characters": [
    "C001",
    "C003"
  ],
  "script_count": 11
}


## 4. 장르별 통계 생성 — 구조 설계(가이드) 방식용

각 장르에서 자주 등장하는 stage, motif, theme, main_character 유형을 집계하여 정적 가이드 파일을 생성합니다.

In [15]:
# ── 장르별 집계 구조 초기화 ──────────────────────────────────────────────────
# stage는 프레임워크마다 레이블이 다르므로(영문 vs 한국어) 프레임워크별로 분리 집계
# unit_motif, theme, work_motif, main_character는 프레임워크 무관 → 통합 집계
genre_stats = defaultdict(lambda: {
    "work_count":             0,
    "work_count_by_framework": Counter(),   # {'storyhelper': N, 'hero_journey': M}
    "scene_count":            0,
    "themes":                 Counter(),
    "work_motifs":            Counter(),
    "main_characters":        Counter(),
    "stages_storyhelper":     Counter(),    # 영문 stage (스토리헬퍼 15단계)
    "stages_hero_journey":    Counter(),    # 한국어 stage (영웅의 12단계)
    "unit_motifs":            Counter(),    # 프레임워크 무관
    "emotions_by_stage":      defaultdict(Counter),  # stage별 감정 집계
})

for work in all_works:
    genres    = work.get("genre", []) or ["기타"]
    framework = get_framework(work.get("structure", ""))

    for genre in genres:
        s = genre_stats[genre]
        s["work_count"] += 1
        s["work_count_by_framework"][framework] += 1
        s["themes"][work.get("theme", "미상")] += 1
        s["work_motifs"][work.get("motif", "미상")] += 1
        s["main_characters"][work.get("main_character", "미상")] += 1

        for unit in work.get("units", []):
            s["scene_count"] += 1
            stage = unit.get("stage", "미상")
            if framework == "hero_journey":
                s["stages_hero_journey"][stage] += 1
            else:
                s["stages_storyhelper"][stage] += 1
            s["unit_motifs"][unit.get("unit_motif", "미상")] += 1
            for script in unit.get("story_scripts", []):
                emotion = script.get("emotion")
                if emotion:
                    s["emotions_by_stage"][stage][emotion] += 1


# ── Top-N 추출 후 직렬화 가능한 dict로 변환 ─────────────────────────────────
TOP_N = 10

genre_stats_serializable = {}
for genre, s in sorted(genre_stats.items()):
    genre_stats_serializable[genre] = {
        "work_count":              s["work_count"],
        "work_count_by_framework": dict(s["work_count_by_framework"]),
        "scene_count":             s["scene_count"],
        # 통합 집계 (프레임워크 무관)
        "top_themes":              s["themes"].most_common(TOP_N),
        "top_work_motifs":         s["work_motifs"].most_common(TOP_N),
        "top_main_characters":     s["main_characters"].most_common(TOP_N),
        "top_unit_motifs":         s["unit_motifs"].most_common(TOP_N),
        # stage별 감정 Top3 (구조 설계 프롬프트용)
        "top_emotions_by_stage":   {
            stage: emo_c.most_common(3)
            for stage, emo_c in s["emotions_by_stage"].items()
        },
        # 프레임워크별 stage 통계 (레이블 체계가 달라 분리)
        "top_stages_storyhelper":  s["stages_storyhelper"].most_common(TOP_N),
        "top_stages_hero_journey": s["stages_hero_journey"].most_common(TOP_N),
    }

# 저장
out_path = OUTPUT_DIR / "genre_stats.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(genre_stats_serializable, f, ensure_ascii=False, indent=2)

print(f"장르 통계 저장 완료: {out_path}")
print(f"총 {len(genre_stats_serializable)}개 장르")
print("\n장르 목록 (작품 수 기준):")
for g, s in sorted(genre_stats_serializable.items(), key=lambda x: -x[1]["work_count"]):
    fw = s["work_count_by_framework"]
    hj = fw.get("hero_journey", 0)
    hj_str = f"  ★hero_journey: {hj}개" if hj > 0 else ""
    print(f"  {g}: 작품 {s['work_count']:,}개, 씬 {s['scene_count']:,}개{hj_str}")

장르 통계 저장 완료: C:\Users\User\Desktop\Github\Dive.ai\output\genre_stats.json
총 10개 장르

장르 목록 (작품 수 기준):
  드라마: 작품 2,264개, 씬 57,291개  ★hero_journey: 21개
  멜로/로맨스: 작품 1,450개, 씬 36,409개  ★hero_journey: 5개
  스릴러: 작품 786개, 씬 20,365개  ★hero_journey: 5개
  판타지: 작품 570개, 씬 12,531개  ★hero_journey: 39개
  액션: 작품 434개, 씬 11,910개  ★hero_journey: 31개
  미스터리: 작품 247개, 씬 6,425개  ★hero_journey: 2개
  코미디: 작품 243개, 씬 5,039개
  SF: 작품 179개, 씬 4,018개  ★hero_journey: 3개
  공포(호러): 작품 100개, 씬 2,830개  ★hero_journey: 2개
  전쟁: 작품 75개, 씬 1,860개  ★hero_journey: 1개


## 5. 장르 통계 미리보기

구조 설계 방식에서 실제로 LLM에 주입할 통계 가이드 내용을 확인합니다.

In [16]:
def print_genre_guide(genre_name: str):
    """특정 장르의 통계 가이드를 출력. 프레임워크별 stage 통계를 구분해서 표시."""
    if genre_name not in genre_stats_serializable:
        print(f"장르 '{genre_name}' 없음. 사용 가능한 장르:")
        print(", ".join(sorted(genre_stats_serializable.keys())))
        return

    s = genre_stats_serializable[genre_name]
    fw = s["work_count_by_framework"]
    print(f"=== [{genre_name}] 장르 통계 가이드 ===")
    print(f"  작품 수: {s['work_count']:,}개  |  씬 수: {s['scene_count']:,}개")
    print(f"  └ storyhelper: {fw.get('storyhelper', 0)}개  |  hero_journey: {fw.get('hero_journey', 0)}개")
    print()

    if s["top_stages_storyhelper"]:
        print(f"  [스토리헬퍼 15단계] Top 서사 단계:")
        for stage, cnt in s["top_stages_storyhelper"]:
            print(f"    {stage}: {cnt}회")
        print()

    if s["top_stages_hero_journey"]:
        print(f"  [영웅의 12단계] Top 서사 단계:")
        for stage, cnt in s["top_stages_hero_journey"]:
            print(f"    {stage}: {cnt}회")
        print()

    print(f"  Top 씬 모티프(unit_motif) — 프레임워크 통합:")
    for motif, cnt in s["top_unit_motifs"]:
        print(f"    {motif}: {cnt}회")
    print()
    print(f"  Top 작품 테마:")
    for theme, cnt in s["top_themes"]:
        print(f"    {theme}: {cnt}회")
    print()
    print(f"  Top 작품 모티프:")
    for motif, cnt in s["top_work_motifs"]:
        print(f"    {motif}: {cnt}회")
    print()
    print(f"  Top 주인공 유형:")
    for mc, cnt in s["top_main_characters"]:
        print(f"    {mc}: {cnt}회")
    print()
    print(f"  서사 단계별 지배 감정 (Top3):")
    stage_key = "top_stages_storyhelper" if s["top_stages_storyhelper"] else "top_stages_hero_journey"
    top_stages = [st for st, _ in s[stage_key][:5]]
    emotions_by_stage = s.get("top_emotions_by_stage", {})
    for stage in top_stages:
        emos = ", ".join(e for e, _ in emotions_by_stage.get(stage, []))
        if emos:
            print(f"    {stage}: {emos}")


print_genre_guide("스릴러")
print()
print_genre_guide("판타지")

=== [스릴러] 장르 통계 가이드 ===
  작품 수: 786개  |  씬 수: 20,365개
  └ storyhelper: 781개  |  hero_journey: 5개

  [스토리헬퍼 15단계] Top 서사 단계:
    Setting-up: 1916회
    Making a Choice: 1748회
    2nd Accident: 1740회
    1st Accident: 1697회
    Villains Move: 1683회
    Doubts & Debate: 1604회
    Choice to Fight: 1518회
    Defeat: 1334회
    Main Character: 1062회
    Innermost Cave: 1061회

  [영웅의 12단계] Top 서사 단계:
    시험, 협력자, 적대자: 18회
    모험에의 소명: 16회
    소명부활의 거부: 16회
    일상세계: 15회
    시련: 13회
    부활: 12회
    첫 관문의 통과: 11회
    동굴 가장 깊은 곳으로의 진입: 11회
    정신적 스승과의 만남: 9회
    보상: 9회

  Top 씬 모티프(unit_motif) — 프레임워크 통합:
    미상: 1408회
    부탁/제안: 636회
    계략을 꾸밈: 610회
    추적/수색: 597회
    상황 파악: 476회
    걱정: 380회
    도움을 줌/받음: 371회
    적과의 대치: 354회
    비밀의 발견/발각: 353회
    죽음: 345회

  Top 작품 테마:
    추적: 133회
    몰락: 104회
    사랑: 91회
    복수: 81회
    구출: 76회
    수수께끼: 65회
    상승: 36회
    지독한 행위: 31회
    희생자: 30회
    탈출: 27회

  Top 작품 모티프:
    연쇄 살인: 48회
    미결의 살인사건: 32회
    납치: 31회
    예언: 29회
    감금: 28회
    누명: 23

## 6. 메타데이터 RAG 시뮬레이션 — 씬 필터링

벡터 DB 구축 전, 장르 + 모티프 키워드 기반으로 유사 씬을 직접 필터링하는 방식으로 메타데이터 RAG를 시뮬레이션합니다.

In [17]:
from typing import Optional, List

def retrieve_scene_metadata(
    genre: str,
    motif_keywords: Optional[List[str]] = None,
    stage: Optional[str] = None,
    framework: Optional[str] = None,
    top_k: int = 5,
) -> List[dict]:
    """장르/모티프/stage 기반으로 씬 메타데이터를 필터링하여 반환.

    Parameters
    ----------
    genre           : 검색할 장르 (예: '스릴러')
    motif_keywords  : 모티프 키워드 목록 — 하나라도 포함되면 포함 (예: ['복수', '배신'])
    stage           : 특정 스토리 단계로 필터 (예: 'Climax')
    framework       : 'storyhelper' | 'hero_journey' | None(전체)
    top_k           : 반환할 최대 씬 수

    Returns
    -------
    List[dict] : 필터링된 씬 메타데이터 목록
    """
    results = []
    for scene in scene_index:
        # 장르 필터
        if genre not in scene["genre"]:
            continue
        # 프레임워크 필터 (선택)
        if framework and scene.get("framework") != framework:
            continue
        # 모티프 키워드 필터 (선택)
        if motif_keywords:
            unit_motif = scene.get("unit_motif") or ""
            work_motif = scene.get("work_motif") or ""
            if not any(kw in unit_motif or kw in work_motif for kw in motif_keywords):
                continue
        # stage 필터 (선택)
        if stage and scene.get("stage") != stage:
            continue

        results.append({
            "work_id":    scene["work_id"],
            "scene_id":   scene["scene_id"],
            "framework":  scene["framework"],
            "stage":      scene["stage"],
            "unit_motif": scene["unit_motif"],
            "storyline":  scene["storyline"],
            "genre":      scene["genre"],
            "theme":      scene["theme"],
        })
        if len(results) >= top_k:
            break

    return results


def build_rag_prompt_context(scenes: List[dict]) -> str:
    """검색된 씬 메타데이터를 LLM 프롬프트 주입용 텍스트로 변환."""
    lines = ["[유사 작품 씬 패턴 참고]"]
    for s in scenes:
        stage     = s.get("stage", "?")
        motif     = s.get("unit_motif", "?")
        storyline = s.get("storyline", "")
        lines.append(f"- [{stage} / {motif}] {storyline}")
    return "\n".join(lines)


# ── 사용 예시 ────────────────────────────────────────────────────────────────
print("=== 스릴러 (storyhelper만) ===")
retrieved = retrieve_scene_metadata(
    genre="스릴러",
    motif_keywords=["복수", "배신", "결단"],
    framework="storyhelper",
    top_k=5,
)
print(f"검색된 씬 수: {len(retrieved)}개\n")
print(build_rag_prompt_context(retrieved))

print("\n=== 판타지 (hero_journey 포함 전체) ===")
retrieved_f = retrieve_scene_metadata(
    genre="판타지",
    motif_keywords=["모험", "성장", "선택"],
    top_k=5,
)
print(f"검색된 씬 수: {len(retrieved_f)}개\n")
print(build_rag_prompt_context(retrieved_f))

=== 스릴러 (storyhelper만) ===
검색된 씬 수: 5개

[유사 작품 씬 패턴 참고]
- [Villains Move / 배신] C001은 호텔 직원 의무실로 연을 찾아가고, 연은 깨어나서 C001의 부축을 받으며 가는데 C003이 이 모습을 발견한다.
- [Trailer Moments / 복수] C010이 죽고 분노한 C003이 C001을 찾아간다.
- [Doubts & Debate / 복수] 아우에 대한 복수로 C001이 C008를 폭행한다.
- [Innermost Cave / 복수] C001이 E006을 찾아가 칼로 찌른다.
- [Trailer Moments / 배신] C028은 C005이 아내를 잡아놓았다는 사실을 알고 분노한다.

=== 판타지 (hero_journey 포함 전체) ===
검색된 씬 수: 5개

[유사 작품 씬 패턴 참고]
- [Making a Choice / 동료가 됨 (전투/모험의 경우)] C001은 C004의 담임 1을 불러 학교의 문제점 개선을 요구한다.
- [Defeat / 동료가 됨 (전투/모험의 경우)] 아이들이 C023을 무찌를 대책을 강구한다.
- [1st Accident / 동료가 됨 (전투/모험의 경우)] C006는 프로젝트를 함께 하게 된 C001, C002, C003, C004을 환영하며 부탁하고 싶은 게 있는지 물어본다.
- [Setting-up / 잘못된 선택] C004는 C006이 앵커가 된 것을 비롯해 모든 것을 신의 탓으로 돌린다.
- [Making a Choice / 동료가 됨 (전투/모험의 경우)] C004, C001, C010, C005, C009가 우주선 안에서 대화를 나눈다.


## 7. 구조 설계 방식 — 장르 가이드 프롬프트 생성

In [18]:
def build_structure_guide_prompt(
    genre: str,
    framework: str = "storyhelper",
    top_n: int = 5,
) -> str:
    """장르별 통계 가이드를 LLM 프롬프트 주입용 텍스트로 변환.

    Parameters
    ----------
    genre     : 장르 이름
    framework : 'storyhelper'(기본) | 'hero_journey'
                판타지처럼 hero_journey 작품이 많은 장르에서 활용
    top_n     : 각 항목 Top N 개수
    """
    if genre not in genre_stats_serializable:
        return f"[오류] '{genre}' 장르 통계 없음."

    s = genre_stats_serializable[genre]

    stage_key = "top_stages_storyhelper" if framework == "storyhelper" else "top_stages_hero_journey"
    stages    = ", ".join(st for st, _ in s[stage_key][:top_n])
    motifs    = ", ".join(mo for mo, _ in s["top_unit_motifs"][:top_n])
    themes    = ", ".join(th for th, _ in s["top_themes"][:top_n])
    w_motifs  = ", ".join(mo for mo, _ in s["top_work_motifs"][:top_n])

    fw_label = "스토리헬퍼 15단계" if framework == "storyhelper" else "영웅의 12단계"

    emotions_by_stage = s.get("top_emotions_by_stage", {})
    top_stage_list    = [st for st, _ in s[stage_key][:top_n]]
    stage_emo_lines   = []
    for st in top_stage_list:
        emos = ", ".join(e for e, _ in emotions_by_stage.get(st, []))
        if emos:
            stage_emo_lines.append(f"  {st}: {emos}")
    stage_emo_str = "\n".join(stage_emo_lines)

    return (
        f"[{genre} 장르 흥행 패턴 가이드 — {fw_label}]\n"
        f"- 핵심 서사 단계: {stages}\n"
        f"- 자주 쓰이는 씬 모티프: {motifs}\n"
        f"- 주요 테마: {themes}\n"
        f"- 작품 모티프: {w_motifs}\n"
        f"- 서사 단계별 지배 감정:\n{stage_emo_str}"
    )


# ── 사용 예시 ────────────────────────────────────────────────────────────────
print(build_structure_guide_prompt("스릴러", framework="storyhelper"))
print()
print(build_structure_guide_prompt("판타지", framework="storyhelper"))
print()
print(build_structure_guide_prompt("판타지", framework="hero_journey"))

[스릴러 장르 흥행 패턴 가이드 — 스토리헬퍼 15단계]
- 핵심 서사 단계: Setting-up, Making a Choice, 2nd Accident, 1st Accident, Villains Move
- 자주 쓰이는 씬 모티프: 미상, 부탁/제안, 계략을 꾸밈, 추적/수색, 상황 파악
- 주요 테마: 추적, 몰락, 사랑, 복수, 구출
- 작품 모티프: 연쇄 살인, 미결의 살인사건, 납치, 예언, 감금

[판타지 장르 흥행 패턴 가이드 — 스토리헬퍼 15단계]
- 핵심 서사 단계: Making a Choice, 2nd Accident, 1st Accident, Setting-up, Doubts & Debate
- 자주 쓰이는 씬 모티프: 미상, 부탁/제안, 정보 획득, 계략을 꾸밈, 도움을 줌/받음
- 주요 테마: 모험, 사랑, 도전, 구출, 성숙
- 작품 모티프: 갑작스런 사고, 소년 영웅, 차원이동, 뜻밖의 초능력, 강제된 과업

[판타지 장르 흥행 패턴 가이드 — 영웅의 12단계]
- 핵심 서사 단계: 시험, 협력자, 적대자, 모험에의 소명, 시련, 일상세계, 동굴 가장 깊은 곳으로의 진입
- 자주 쓰이는 씬 모티프: 미상, 부탁/제안, 정보 획득, 계략을 꾸밈, 도움을 줌/받음
- 주요 테마: 모험, 사랑, 도전, 구출, 성숙
- 작품 모티프: 갑작스런 사고, 소년 영웅, 차원이동, 뜻밖의 초능력, 강제된 과업


## 8. 두 방식 나란히 비교 — 동일 소재 입력 시뮬레이션

> 실제 LLM 호출 없이, 각 방식이 프롬프트에 주입할 컨텍스트가 어떻게 다른지 확인합니다.

In [19]:
USER_INPUT = "배신당한 복수극"
USER_GENRE = "스릴러"
MOTIF_HINTS = ["복수", "배신", "결단", "역전"]

print("=" * 60)
print(f"[유저 입력] {USER_INPUT}  (장르: {USER_GENRE})")
print("=" * 60)

print("\n🅐 메타데이터 RAG 방식 — 프롬프트 컨텍스트")
print("-" * 60)
rag_scenes = retrieve_scene_metadata(USER_GENRE, motif_keywords=MOTIF_HINTS, top_k=5)
print(build_rag_prompt_context(rag_scenes))

print("\n🅑 구조 설계 방식 — 프롬프트 컨텍스트")
print("-" * 60)
print(build_structure_guide_prompt(USER_GENRE))

[유저 입력] 배신당한 복수극  (장르: 스릴러)

🅐 메타데이터 RAG 방식 — 프롬프트 컨텍스트
------------------------------------------------------------
[유사 작품 씬 패턴 참고]
- [Villains Move / 배신] C001은 호텔 직원 의무실로 연을 찾아가고, 연은 깨어나서 C001의 부축을 받으며 가는데 C003이 이 모습을 발견한다.
- [Trailer Moments / 복수] C010이 죽고 분노한 C003이 C001을 찾아간다.
- [Doubts & Debate / 복수] 아우에 대한 복수로 C001이 C008를 폭행한다.
- [Innermost Cave / 복수] C001이 E006을 찾아가 칼로 찌른다.
- [Trailer Moments / 배신] C028은 C005이 아내를 잡아놓았다는 사실을 알고 분노한다.

🅑 구조 설계 방식 — 프롬프트 컨텍스트
------------------------------------------------------------
[스릴러 장르 흥행 패턴 가이드 — 스토리헬퍼 15단계]
- 핵심 서사 단계: Setting-up, Making a Choice, 2nd Accident, 1st Accident, Villains Move
- 자주 쓰이는 씬 모티프: 미상, 부탁/제안, 계략을 꾸밈, 추적/수색, 상황 파악
- 주요 테마: 추적, 몰락, 사랑, 복수, 구출
- 작품 모티프: 연쇄 살인, 미결의 살인사건, 납치, 예언, 감금


---

# 동아시아 고전 데이터 전처리

> 유저가 **"조선", "중국 무협", "일본 설화"** 등 고전 장르를 명시적으로 선택했을 때만 보조 참조합니다.  
> 문화콘텐츠 데이터와 달리 **단락(txt 파일 1개 = 단락 1개)** 단위로 처리합니다.

## 10. 고전 단락 인덱스 생성

In [20]:
def iter_all_classic_labels():
    """Training + Validation 전체 고전 라벨링 데이터를 순회하는 제너레이터.

    Yields
    ------
    (file_stem, split, country_name, label_dict)
    """
    for split in ["Training", "Validation"]:
        prefix = "TL" if split == "Training" else "VL"
        for country_code, country_name in COUNTRY_MAP.items():
            max_group = COUNTRY_GROUPS[country_name]
            for g in range(1, max_group + 1):
                lbl_dir = CLASSIC_BASE_DIR / split / "02.라벨링데이터" / f"{prefix}_ST01_{country_name}_{g:02d}_info"
                if not lbl_dir.exists():
                    continue
                for json_file in sorted(lbl_dir.glob("*_info.json")):
                    stem = json_file.stem
                    file_stem = stem[:-5] if stem.endswith("_info") else stem
                    try:
                        with open(json_file, encoding="utf-8") as f:
                            label = json.load(f)
                        yield file_stem, split, country_name, label
                    except Exception as e:
                        print(f"[경고] 로드 실패: {json_file} — {e}")


# ── 단락 인덱스 생성 ──────────────────────────────────────────────────────────
classic_index = []

for file_stem, split, country_name, label in iter_all_classic_labels():
    classification = label.get("내용분류", {})

    # 파일명에서 work_id 추출: ST01_{country}_{class}_{work}_{sec}_{para}
    parts = file_stem.split("_")
    work_id = f"{parts[1]}_{parts[2]}_{parts[3]}" if len(parts) >= 4 else file_stem

    classic_index.append({
        "paragraph_id":  file_stem,
        "work_id":       work_id,
        "country":       country_name,
        "country_code":  label.get("country", ""),
        "split":         split,
        # ── 내용 분류 레이블 ────────────────────────────────────────────────
        "genre":         classification.get("장르", []),
        "motif":         classification.get("사건/모티프", []),
        "emotion":       classification.get("감정", []),
        "action":        classification.get("행동", []),
        "space":         classification.get("공간", []),
        "character":     classification.get("캐릭터", []),
        "occupation":    classification.get("직업/신분", []),
        "dialogue_type": classification.get("대화형태", []),
        "relation":      classification.get("인물관계", []),
        # ── 주제 정보 ────────────────────────────────────────────────────────
        "keywords":      label.get("주제어", []),
        "summary":       label.get("주제문", ""),
    })

# 저장
out_path = OUTPUT_DIR / "classic_paragraph_index.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(classic_index, f, ensure_ascii=False, indent=2)

# 국가별 단락 수 확인
country_counter = Counter(p["country"] for p in classic_index)
print(f"고전 단락 인덱스 저장 완료: {out_path}")
print(f"총 {len(classic_index):,}개 단락")
print(f"\n국가별:")
for country in ["한국", "중국", "일본"]:
    print(f"  {country}: {country_counter[country]:,}개")
print("\n샘플:")
print(json.dumps(classic_index[0], ensure_ascii=False, indent=2))

고전 단락 인덱스 저장 완료: C:\Users\User\Desktop\Github\Dive.ai\output\classic_paragraph_index.json
총 12,717개 단락

국가별:
  한국: 6,304개
  중국: 3,257개
  일본: 3,156개

샘플:
{
  "paragraph_id": "ST01_01_01_0073_04_0145",
  "work_id": "01_01_0073",
  "country": "한국",
  "country_code": "01",
  "split": "Training",
  "genre": [
    "가문소설"
  ],
  "motif": [
    "전쟁"
  ],
  "emotion": [
    "기쁨"
  ],
  "action": [
    "반응"
  ],
  "space": [
    "궁"
  ],
  "character": [
    "왕족/귀족"
  ],
  "occupation": [
    "왕",
    "신하"
  ],
  "dialogue_type": [
    "명령",
    "설명",
    "답변"
  ],
  "relation": [
    "군신"
  ],
  "keywords": [
    "손무자",
    "여병",
    "궁녀",
    "군사",
    "진법"
  ],
  "summary": "연왕후가 모란을 죽이기 위해 궁녀를 병사로 삼겠다고 하고 상께서 궁녀를 모았지만 그 중에 모란이 보이지 않았다."
}


## 11. 국가별 통계 생성

In [21]:
# ── 국가별 집계 ───────────────────────────────────────────────────────────────
classic_stats = defaultdict(lambda: {
    "paragraph_count": 0,
    "work_ids":        set(),
    "genres":          Counter(),
    "motifs":          Counter(),
    "spaces":          Counter(),
    "characters":      Counter(),
    "emotions":        Counter(),
    "occupations":     Counter(),
    "dialogue_types":  Counter(),
    "relations":       Counter(),
})

for p in classic_index:
    country = p["country"]
    s = classic_stats[country]
    s["paragraph_count"] += 1
    s["work_ids"].add(p["work_id"])
    for g  in p["genre"]:         s["genres"][g] += 1
    for m  in p["motif"]:         s["motifs"][m] += 1
    for sp in p["space"]:         s["spaces"][sp] += 1
    for c  in p["character"]:     s["characters"][c] += 1
    for e  in p["emotion"]:       s["emotions"][e] += 1
    for o  in p["occupation"]:    s["occupations"][o] += 1
    for d  in p["dialogue_type"]: s["dialogue_types"][d] += 1
    for r  in p["relation"]:      s["relations"][r] += 1

TOP_N = 10

classic_stats_serializable = {}
for country in ["한국", "중국", "일본"]:
    if country not in classic_stats:
        continue
    s = classic_stats[country]
    classic_stats_serializable[country] = {
        "paragraph_count": s["paragraph_count"],
        "work_count":      len(s["work_ids"]),
        "top_genres":      s["genres"].most_common(TOP_N),
        "top_motifs":      s["motifs"].most_common(TOP_N),
        "top_spaces":      s["spaces"].most_common(TOP_N),
        "top_characters":  s["characters"].most_common(TOP_N),
        "top_emotions":    s["emotions"].most_common(TOP_N),
        "top_occupations": s["occupations"].most_common(TOP_N),
        "top_dialogue_types": s["dialogue_types"].most_common(TOP_N),
        "top_relations":   s["relations"].most_common(TOP_N),
    }

# 저장
out_path = OUTPUT_DIR / "classic_country_stats.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(classic_stats_serializable, f, ensure_ascii=False, indent=2)

print(f"국가별 통계 저장 완료: {out_path}")
print()
for country, s in classic_stats_serializable.items():
    print(f"=== [{country}] ===")
    print(f"  작품 수: {s['work_count']}개  |  단락 수: {s['paragraph_count']:,}개")
    print(f"  Top 장르:   {[g for g, _ in s['top_genres'][:5]]}")
    print(f"  Top 모티프: {[m for m, _ in s['top_motifs'][:5]]}")
    print(f"  Top 공간:   {[sp for sp, _ in s['top_spaces'][:5]]}")
    print()

국가별 통계 저장 완료: C:\Users\User\Desktop\Github\Dive.ai\output\classic_country_stats.json

=== [한국] ===
  작품 수: 71개  |  단락 수: 6,304개
  Top 장르:   ['가문소설', '판타지', '로맨스', '영웅소설', '가사']
  Top 모티프: ['해당없음', '전쟁', '운명', '사랑', '적강']
  Top 공간:   ['가옥', '궁', '해당없음', '자연', '도성']

=== [중국] ===
  작품 수: 319개  |  단락 수: 3,257개
  Top 장르:   ['해당없음', '판타지', '로맨스', '무협', '호러']
  Top 모티프: ['해당없음', '기연/행운', '초능력', '죽음/부활', '마법']
  Top 공간:   ['해당없음', '가옥', '자연', '도성', '궁']

=== [일본] ===
  작품 수: 426개  |  단락 수: 3,156개
  Top 장르:   ['설화', '미스터리', '호러', '편지소설', '해당없음']
  Top 모티프: ['종교', '기연/행운', '죽음/부활', '불운', '보상/징벌']
  Top 공간:   ['가옥', '자연', '절/신당', '일본', '해당없음']



## 12. 고전 장르 RAG 시뮬레이션

In [22]:
def retrieve_classic_paragraph(
    country: Optional[str] = None,
    genre_keywords: Optional[List[str]] = None,
    motif_keywords: Optional[List[str]] = None,
    space: Optional[str] = None,
    top_k: int = 5,
) -> List[dict]:
    """국가/장르/모티프/공간 기반으로 고전 단락을 필터링하여 반환.

    Parameters
    ----------
    country         : '한국' | '중국' | '일본' | None(전체)
    genre_keywords  : 장르 키워드 목록 (예: ['설화', '무협'])
    motif_keywords  : 모티프 키워드 목록 (예: ['복수', '기적'])
    space           : 배경 공간 필터 (예: '궁')
    top_k           : 반환할 최대 단락 수

    Returns
    -------
    List[dict] : 필터링된 단락 메타데이터 목록
    """
    results = []
    for p in classic_index:
        # 국가 필터
        if country and p["country"] != country:
            continue
        # 장르 키워드 필터
        if genre_keywords:
            if not any(kw in g for g in p["genre"] for kw in genre_keywords):
                continue
        # 모티프 키워드 필터
        if motif_keywords:
            if not any(kw in m for m in p["motif"] for kw in motif_keywords):
                continue
        # 공간 필터
        if space and space not in p["space"]:
            continue

        results.append({
            "paragraph_id": p["paragraph_id"],
            "country":      p["country"],
            "genre":        p["genre"],
            "motif":        p["motif"],
            "space":        p["space"],
            "character":    p["character"],
            "keywords":     p["keywords"],
            "summary":      p["summary"],
        })
        if len(results) >= top_k:
            break

    return results


def build_classic_rag_context(paragraphs: List[dict]) -> str:
    """검색된 고전 단락 메타데이터를 LLM 프롬프트 주입용 텍스트로 변환."""
    lines = ["[동아시아 고전 참고 — 세계관·분위기]"]
    for p in paragraphs:
        country  = p["country"]
        motif    = ", ".join(p["motif"][:3]) if p["motif"] else "?"
        space    = ", ".join(p["space"][:2]) if p["space"] else "?"
        summary  = p["summary"]
        lines.append(f"- [{country} / {motif} / {space}] {summary}")
    return "\n".join(lines)


# ── 사용 예시 ────────────────────────────────────────────────────────────────
print("=== 조선 퇴마사 소재 — 한국 설화 검색 ===")
result_kr = retrieve_classic_paragraph(
    country="한국",
    genre_keywords=["설화"],
    motif_keywords=["귀신", "요괴", "기적"],
    top_k=5,
)
print(f"검색된 단락 수: {len(result_kr)}개\n")
print(build_classic_rag_context(result_kr))

print("\n=== 중국 무협 소재 — 중국 검색 ===")
result_cn = retrieve_classic_paragraph(
    country="중국",
    genre_keywords=["무협"],
    top_k=5,
)
print(f"검색된 단락 수: {len(result_cn)}개\n")
print(build_classic_rag_context(result_cn))

print("\n=== 일본 설화 — 궁 배경 검색 ===")
result_jp = retrieve_classic_paragraph(
    country="일본",
    genre_keywords=["설화"],
    space="궁",
    top_k=5,
)
print(f"검색된 단락 수: {len(result_jp)}개\n")
print(build_classic_rag_context(result_jp))

=== 조선 퇴마사 소재 — 한국 설화 검색 ===
검색된 단락 수: 0개

[동아시아 고전 참고 — 세계관·분위기]

=== 중국 무협 소재 — 중국 검색 ===
검색된 단락 수: 5개

[동아시아 고전 참고 — 세계관·분위기]
- [중국 / 해당없음 / 해당없음] 회음사에 남긴 시를 보면 유방이 중원을 차지했을 당시의 긴박한 상황을 알 수 있다.
- [중국 / 기연/행운 / 해당없음] 연소왕은 검은 조개에서 나온 진주를 통해 여름의 더위를 식히며 그 가치를 높이 평가하였다.
- [중국 / 해당없음 / 해당없음] 촉 지역의 석순가에서는 여름철 큰 비가 내린 후 다양한 색상의 작은 진주를 주울 수 있다는 이야기가 전해진다.
- [중국 / 해당없음 / 도성, 궁] 무측천 시절에 서번의 나라가 바친 기이한 유물들 중 청니주가 포함되어 있었으나, 무측천은 그 가치를 제대로 인식하지 못했다.
- [중국 / 해당없음 / 절/신당] 승려가 진주를 팔고 호인이 높은 가격에 구매한 사건은 결국 무측천에게 보고되었다.

=== 일본 설화 — 궁 배경 검색 ===
검색된 단락 수: 5개

[동아시아 고전 참고 — 세계관·분위기]
- [일본 / 기적, 종교 / 궁] 에이지쓰가 내전으로 향하며 법화경을 읽자 천황의 병이 호전되어 승강으로 임명될 뻔했지만 에이지쓰는 이를 거절하고 퇴장했다.
- [일본 / 악기 연주 / 궁] 미나모토노 히로마사는 특히 관현의 도에 통달했던 인물인데, 무라카미 천황의 치세에 궁중의 전상인이었다.
- [일본 / 악기 연주 / 궁, 절/신당] 오사카의 관문에 살고 있는 맹인 세미마로는 아쓰미 친왕의 잡색으로, 관현의 도를 깊이 연구했으며 비파 연주에 능숙해지게 되었다.
- [일본 / 노래 짓기 / 궁, 가옥] 이치조인 천황의 치세에 긴토 대납언이 노래를 지어 바친 이야기를 다룬다.
- [일본 / 재촉, 기다림 / 궁] 대납언이 궁중에 입궐하지 않아 관백이 초조해하는 상황에서 다른 사람들의 기대가 모아지고 있다.


## 13. 통합 프롬프트 시뮬레이션 — 고전 장르 선택 시

> 문화콘텐츠(서사 구조) + 동아시아 고전(세계관·분위기)이 실제 LLM 프롬프트에서  
> 어떻게 결합되는지 확인합니다. 고전 장르 여부에 따라 결과가 달라집니다.

In [23]:
def build_combined_prompt_context(
    user_input: str,
    genre: str,
    background: str,
    motif_hints: Optional[List[str]] = None,
    scene_top_k: int = 5,
    classic_top_k: int = 3,
) -> str:
    """문화콘텐츠 서사 구조 + 동아시아 고전 세계관을 결합한 프롬프트 컨텍스트 생성.

    Parameters
    ----------
    user_input   : 유저 입력 소재
    genre        : 장르 (예: '스릴러')
    background   : 유저 선택 배경 (예: '조선', '현대')
    motif_hints  : 모티프 힌트 키워드 목록
    scene_top_k  : 문화콘텐츠 씬 검색 수
    classic_top_k: 고전 단락 검색 수
    """
    lines = [
        f"[유저 입력] {user_input}  (장르: {genre}, 배경: {background})",
        "",
    ]

    # ① 문화콘텐츠 — 서사 구조 (항상 실행)
    scenes = retrieve_scene_metadata(genre, motif_keywords=motif_hints, top_k=scene_top_k)
    lines.append("[흥행 서사 구조 참고 — 스토리헬퍼 15단계]  ← 문화콘텐츠 데이터 (항상 사용)")
    for s in scenes:
        stage   = s.get("stage", "?")
        motif   = s.get("unit_motif", "?")
        story   = s.get("storyline", "")
        lines.append(f"- [{stage} / {motif}] {story}")

    # ② 동아시아 고전 — 세계관·분위기 (고전 장르 선택 시에만)
    lines.append("")
    if is_classic_genre(background):
        # 배경 → 국가 매핑
        bg_to_country = {"조선": "한국", "한국설화": "한국", "중국무협": "중국", "일본설화": "일본"}
        country = next((v for k, v in bg_to_country.items() if k in background), None)
        classics = retrieve_classic_paragraph(
            country=country,
            motif_keywords=motif_hints,
            top_k=classic_top_k,
        )
        lines.append("[동아시아 고전 참고 — 세계관·분위기]  ← 고전 데이터 (고전 장르 선택 시에만)")
        for p in classics:
            motif = ", ".join(p["motif"][:3]) if p["motif"] else "?"
            space = ", ".join(p["space"][:2]) if p["space"] else "?"
            lines.append(f"- [{p['country']} / {motif} / {space}] {p['summary']}")
    else:
        lines.append(f"[고전 데이터 미참조 — 비고전 배경 '{background}']")

    return "\n".join(lines)


# ── 시뮬레이션 ────────────────────────────────────────────────────────────────
SEP = "=" * 65

print(SEP)
print("▶ 케이스 A: 조선 배경 (고전 장르 ON)")
print(SEP)
print(build_combined_prompt_context(
    user_input="퇴마사 복수극",
    genre="스릴러",
    background="조선",
    motif_hints=["복수", "배신", "귀신"],
))

print()
print(SEP)
print("▶ 케이스 B: 현대 배경 (고전 장르 OFF)")
print(SEP)
print(build_combined_prompt_context(
    user_input="퇴마사 복수극",
    genre="스릴러",
    background="현대",
    motif_hints=["복수", "배신"],
))

print()
print(SEP)
print("▶ 케이스 C: 중국 무협 배경 (고전 장르 ON)")
print(SEP)
print(build_combined_prompt_context(
    user_input="협객의 의리와 배신",
    genre="액션",
    background="중국무협",
    motif_hints=["배신", "의리", "결투"],
))

▶ 케이스 A: 조선 배경 (고전 장르 ON)
[유저 입력] 퇴마사 복수극  (장르: 스릴러, 배경: 조선)

[흥행 서사 구조 참고 — 스토리헬퍼 15단계]  ← 문화콘텐츠 데이터 (항상 사용)
- [Villains Move / 배신] C001은 호텔 직원 의무실로 연을 찾아가고, 연은 깨어나서 C001의 부축을 받으며 가는데 C003이 이 모습을 발견한다.
- [Trailer Moments / 복수] C010이 죽고 분노한 C003이 C001을 찾아간다.
- [Doubts & Debate / 복수] 아우에 대한 복수로 C001이 C008를 폭행한다.
- [Innermost Cave / 복수] C001이 E006을 찾아가 칼로 찌른다.
- [Trailer Moments / 배신] C028은 C005이 아내를 잡아놓았다는 사실을 알고 분노한다.

[동아시아 고전 참고 — 세계관·분위기]  ← 고전 데이터 (고전 장르 선택 시에만)
- [한국 / 복수, 운명, 보상/징벌 / 궁] 모란을 베어 군대 안에서 호령한 후, 모든 궁녀들과 삼군이 아연실색하고 진법 연습이 법칙에 맞게 이루어졌다.
- [한국 / 복수, 운명, 보상/징벌 / 궁] 하태후의 분노로 인해 두 사람은 태액지에 가둬지게 되었다. 또한 두 사람이 달아난 것 처럼 꾸며졌다.
- [한국 / 복수, 운명, 속임수 / 궁] 황후께서 유연 후 부인 양씨를 사랑하여 궐내에 두었지만 비극적인 사고로 인해 슬픔이 가득한 상황이다.

▶ 케이스 B: 현대 배경 (고전 장르 OFF)
[유저 입력] 퇴마사 복수극  (장르: 스릴러, 배경: 현대)

[흥행 서사 구조 참고 — 스토리헬퍼 15단계]  ← 문화콘텐츠 데이터 (항상 사용)
- [Villains Move / 배신] C001은 호텔 직원 의무실로 연을 찾아가고, 연은 깨어나서 C001의 부축을 받으며 가는데 C003이 이 모습을 발견한다.
- [Trailer Moments / 복수] C010이 죽고 분노한 C003이 C001을 찾아간다.
- [Doubts & De

## 9. 전처리 결과 요약

In [24]:
output_files = list(OUTPUT_DIR.glob("*.json"))

print("=== 전처리 완료 ===")
print(f"\n[문화콘텐츠 스토리 데이터]")
print(f"  총 작품 수: {len(all_works):,}개")
print(f"  총 씬 수:   {len(scene_index):,}개")
print(f"  장르 수:    {len(genre_stats_serializable)}개")

print(f"\n[동아시아 고전 데이터]")
print(f"  총 단락 수: {len(classic_index):,}개")
for country, s in classic_stats_serializable.items():
    print(f"  {country}: 작품 {s['work_count']}개 / 단락 {s['paragraph_count']:,}개")

print("\n=== 생성된 파일 ===")
for f in sorted(output_files):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:,.1f} KB")

print("\n=== 파일별 용도 ===")
print("  [문화콘텐츠]")
print("  scene_metadata_index.json  →  메타데이터 RAG 필터링 / 추후 벡터 DB 임베딩 소스")
print("  work_metadata_index.json   →  작품 단위 탐색 및 필터링")
print("  genre_stats.json           →  구조 설계 방식 통계 가이드 프롬프트")
print("  [동아시아 고전]")
print("  classic_paragraph_index.json  →  고전 장르 선택 시 RAG 검색 소스")
print("  classic_country_stats.json    →  국가별 장르·모티프·공간 통계")

=== 전처리 완료 ===

[문화콘텐츠 스토리 데이터]
  총 작품 수: 3,561개
  총 씬 수:   89,851개
  장르 수:    10개

[동아시아 고전 데이터]
  총 단락 수: 12,717개
  한국: 작품 71개 / 단락 6,304개
  중국: 작품 319개 / 단락 3,257개
  일본: 작품 426개 / 단락 3,156개

=== 생성된 파일 ===
  classic_country_stats.json: 13.2 KB
  classic_paragraph_index.json: 11,574.2 KB
  genre_stats.json: 36.6 KB
  scene_metadata_index.json: 57,559.8 KB
  work_metadata_index.json: 3,391.5 KB

=== 파일별 용도 ===
  [문화콘텐츠]
  scene_metadata_index.json  →  메타데이터 RAG 필터링 / 추후 벡터 DB 임베딩 소스
  work_metadata_index.json   →  작품 단위 탐색 및 필터링
  genre_stats.json           →  구조 설계 방식 통계 가이드 프롬프트
  [동아시아 고전]
  classic_paragraph_index.json  →  고전 장르 선택 시 RAG 검색 소스
  classic_country_stats.json    →  국가별 장르·모티프·공간 통계
